# Laboratorio 8 - Filtros FIR com Janelas

Este notebook implementa os exercicios do Laboratorio 8 sobre filtros FIR projetados por truncamento da resposta impulsiva ideal e por janelas prontas do Python.

Um filtro FIR possui resposta impulsiva finita. Por isso, ele e sempre estavel e pode ter fase linear quando os coeficientes sao simetricos.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal


def centered_index(numtaps: int) -> np.ndarray:
    n = np.arange(numtaps)
    return n - (numtaps - 1) / 2


def ideal_lowpass(numtaps: int, wc: float) -> np.ndarray:
    m = centered_index(numtaps)
    h = np.empty(numtaps, dtype=float)
    center = np.isclose(m, 0.0)
    h[center] = wc / np.pi
    h[~center] = np.sin(wc * m[~center]) / (np.pi * m[~center])
    return h


def ideal_highpass(numtaps: int, wc: float) -> np.ndarray:
    impulse = np.zeros(numtaps)
    impulse[numtaps // 2] = 1.0
    return impulse - ideal_lowpass(numtaps, wc)


def ideal_bandpass(numtaps: int, w1: float, w2: float) -> np.ndarray:
    return ideal_lowpass(numtaps, w2) - ideal_lowpass(numtaps, w1)


def ideal_bandstop(numtaps: int, w1: float, w2: float) -> np.ndarray:
    impulse = np.zeros(numtaps)
    impulse[numtaps // 2] = 1.0
    return impulse - ideal_bandpass(numtaps, w1, w2)


def plot_impulse_response(h: np.ndarray, title: str) -> None:
    plt.figure(figsize=(10, 4))
    plt.stem(np.arange(len(h)), h)
    plt.title(title)
    plt.xlabel("n")
    plt.ylabel("h[n]")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_magnitude(h: np.ndarray, title: str, db: bool = False, label: str | None = None) -> None:
    w, response = signal.freqz(h, worN=4096)
    magnitude = np.abs(response)
    y = 20 * np.log10(np.maximum(magnitude, 1e-8)) if db else magnitude
    ylabel = "Modulo (dB)" if db else "Modulo"

    plt.plot(w / np.pi, y, label=label)
    plt.title(title)
    plt.xlabel("Frequencia normalizada (x pi rad/amostra)")
    plt.ylabel(ylabel)
    plt.grid(True)


def show_filter(h: np.ndarray, title: str, db: bool = False) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].stem(np.arange(len(h)), h)
    axes[0].set_title("Resposta impulsiva")
    axes[0].set_xlabel("n")
    axes[0].set_ylabel("h[n]")
    axes[0].grid(True)

    w, response = signal.freqz(h, worN=4096)
    magnitude = np.abs(response)
    y = 20 * np.log10(np.maximum(magnitude, 1e-8)) if db else magnitude
    axes[1].plot(w / np.pi, y)
    axes[1].set_title("Resposta em frequencia")
    axes[1].set_xlabel("Frequencia normalizada (x pi rad/amostra)")
    axes[1].set_ylabel("Modulo (dB)" if db else "Modulo")
    axes[1].grid(True)

    fig.suptitle(title, y=1.03)
    plt.tight_layout()
    plt.show()


## Parte I - Projeto manual por resposta ideal truncada

No item 1, o enunciado usa $F_s = 5000$ Hz e $F_p = 2000$ Hz. A frequencia digital de corte fica:

$$\omega_c = 2\pi \frac{F_p}{F_s} = 2\pi\frac{2000}{5000} = 0.8\pi$$

A resposta impulsiva ideal do passa-baixa e:

$$h_d[n] = \frac{\sin(\omega_c m)}{\pi m}, \quad m = n - \frac{N-1}{2}$$

Na pratica, ao limitar essa resposta para apenas $N$ amostras, estamos multiplicando por uma janela retangular.

In [ ]:
fs = 5000
fp = 2000
wc = 2 * np.pi * fp / fs

for numtaps in [10, 100, 1000]:
    h = ideal_lowpass(numtaps, wc)
    show_filter(h, title=f"Passa-baixa FIR manual | N = {numtaps} | wc = 0.8*pi")


### Analise do item 1

Ao aumentar $N$ de 10 para 100 e 1000, a resposta impulsiva fica mais longa e a resposta em frequencia se aproxima melhor do filtro ideal. A transicao entre passa-faixa e rejeita-faixa fica mais estreita.

A janela usada nesse codigo e a **janela retangular**, tambem chamada de `boxcar`, porque a resposta ideal foi apenas truncada em $N$ amostras sem multiplicar por outra janela.

## Itens 2, 3, 4 e 5 - Passa-alta, passa-banda e rejeita-banda manuais

Para os filtros abaixo foi usado $N = 101$ para ter uma amostra central bem definida. As frequencias do enunciado foram interpretadas como normalizadas por $\pi$:

- Passa-alta: $\omega_c = 0.6\pi$
- Passa-banda: $\omega_1 = 0.2\pi$ e $\omega_2 = 0.5\pi$
- Rejeita-banda do item 5: $\omega_1 = \pi/3$ e $\omega_2 = 2\pi/3$


In [ ]:
numtaps_manual = 101

h_highpass = ideal_highpass(numtaps_manual, 0.6 * np.pi)
h_bandpass = ideal_bandpass(numtaps_manual, 0.2 * np.pi, 0.5 * np.pi)
h_bandstop = ideal_bandstop(numtaps_manual, np.pi / 3, 2 * np.pi / 3)

plt.figure(figsize=(11, 5))
plot_magnitude(h_highpass, "Filtros FIR manuais - modulo", label="Passa-alta wc=0.6*pi")
plot_magnitude(h_bandpass, "Filtros FIR manuais - modulo", label="Passa-banda 0.2*pi a 0.5*pi")
plot_magnitude(h_bandstop, "Filtros FIR manuais - modulo", label="Rejeita-banda pi/3 a 2*pi/3")
plt.legend()
plt.tight_layout()
plt.show()

show_filter(h_highpass, "Passa-alta FIR manual | wc = 0.6*pi")
show_filter(h_bandpass, "Passa-banda FIR manual | w1 = 0.2*pi | w2 = 0.5*pi")
show_filter(h_bandstop, "Rejeita-banda FIR manual | w1 = pi/3 | w2 = 2*pi/3")


### Equacao usada no filtro rejeita-banda ideal

O rejeita-banda ideal pode ser visto como um impulso menos um passa-banda ideal:

$$h_{rb}[n] = \delta[m] - \frac{\sin(\omega_2 m) - \sin(\omega_1 m)}{\pi m}$$

com:

$$m = n - \frac{N-1}{2}$$

No centro, quando $m = 0$, usa-se o limite:

$$h_{rb}[centro] = 1 - \frac{\omega_2 - \omega_1}{\pi}$$

Esse filtro deixa passar baixas e altas frequencias, mas rejeita a faixa entre $\omega_1$ e $\omega_2$.

## Item 5 - Rejeita-faixa ideal com $\pi/3$ e $2\pi/3$

O enunciado pede um filtro rejeita-faixa com ganho 1 fora da faixa $\pi/3 \leq |\omega| \leq 2\pi/3$ e ganho 0 dentro dela. Abaixo, o filtro FIR projetado por truncamento e comparado com a resposta ideal.

![Resposta em frequencia do item 5](item5_rejeita_faixa.png)

In [ ]:
numtaps_item5 = 101
w1_item5 = np.pi / 3
w2_item5 = 2 * np.pi / 3

h_item5 = ideal_bandstop(numtaps_item5, w1_item5, w2_item5)
w, response = signal.freqz(h_item5, worN=4096)
w_norm = w / np.pi
magnitude = np.abs(response)

ideal_response = np.ones_like(w_norm)
ideal_response[(w_norm >= 1 / 3) & (w_norm <= 2 / 3)] = 0

plt.figure(figsize=(11, 5))
plt.plot(w_norm, magnitude, label="FIR projetado")
plt.step(w_norm, ideal_response, where="post", linestyle="--", label="Resposta ideal")
plt.axvline(1 / 3, color="black", linestyle=":", linewidth=1)
plt.axvline(2 / 3, color="black", linestyle=":", linewidth=1)
plt.title("Item 5 - Filtro rejeita-faixa | pi/3 a 2*pi/3")
plt.xlabel("Frequencia normalizada (x pi rad/amostra)")
plt.ylabel("Modulo")
plt.ylim(-0.1, 1.2)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

show_filter(h_item5, "Item 5 - Resposta impulsiva e frequencia | rejeita pi/3 a 2*pi/3")


## Parte II - Usando `signal.firwin`

A funcao `signal.firwin` projeta filtros FIR usando janelas prontas. No exemplo do enunciado:

- `numtaps = 40`: o filtro tem 40 coeficientes. Pela definicao classica, a ordem e `numtaps - 1 = 39`.
- `wc = 0.2`: frequencia de corte normalizada. Como `fs` nao foi informado, `1.0` representa a frequencia de Nyquist.
- Janelas: `boxcar`, `hamming` e `blackman`.


In [ ]:
numtaps = 40
wc_norm = 0.2

windows = {
    "boxcar": signal.firwin(numtaps, wc_norm, window="boxcar"),
    "hamming": signal.firwin(numtaps, wc_norm, window="hamming"),
    "blackman": signal.firwin(numtaps, wc_norm, window="blackman"),
}

plt.figure(figsize=(11, 5))
for name, coeffs in windows.items():
    plot_magnitude(coeffs, "Resposta em frequencia - FIR passa-baixa | numtaps = 40", db=True, label=name)
plt.ylim(-120, 5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Numero de coeficientes: {numtaps}")
print(f"Ordem classica do FIR: {numtaps - 1}")
print(f"Frequencia de corte normalizada: {wc_norm}")


### Analise das janelas

A janela `boxcar` e a retangular: costuma ter transicao mais estreita, mas lobulos laterais maiores, ou seja, pior atenuacao fora da banda.

A janela `hamming` reduz os lobulos laterais em relacao a retangular, melhorando a atenuacao, mas deixa a faixa de transicao um pouco mais larga.

A janela `blackman` reduz ainda mais os lobulos laterais, mas normalmente aumenta mais a largura da transicao. Em filtros, isso mostra o compromisso entre seletividade e atenuacao.

## Item 2 - Alterando para `numtaps = 400`

Agora o mesmo passa-baixa e projetado com 400 coeficientes.

In [ ]:
numtaps = 400
windows_400 = {
    "boxcar": signal.firwin(numtaps, wc_norm, window="boxcar"),
    "hamming": signal.firwin(numtaps, wc_norm, window="hamming"),
    "blackman": signal.firwin(numtaps, wc_norm, window="blackman"),
}

plt.figure(figsize=(11, 5))
for name, coeffs in windows_400.items():
    plot_magnitude(coeffs, "Resposta em frequencia - FIR passa-baixa | numtaps = 400", db=True, label=name)
plt.ylim(-140, 5)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Numero de coeficientes: {numtaps}")
print(f"Ordem classica do FIR: {numtaps - 1}")


### Analise do `numtaps = 400`

Com mais coeficientes, o filtro fica mais seletivo. A transicao ao redor da frequencia de corte fica mais estreita e a resposta se aproxima mais do comportamento ideal. O custo e maior processamento e maior atraso, pois o filtro precisa de mais amostras para calcular cada saida.

## Item 3 - Outros filtros com `firwin`

A seguir sao projetados filtros passa-alta, passa-banda e rejeita-faixa usando a janela de Hamming.

In [ ]:
numtaps = 101

firwin_filters = {
    "passa-alta cutoff=0.35": signal.firwin(numtaps, 0.35, pass_zero=False, window="hamming"),
    "passa-banda 0.25 a 0.55": signal.firwin(numtaps, [0.25, 0.55], pass_zero=False, window="hamming"),
    "rejeita-faixa 0.25 a 0.55": signal.firwin(numtaps, [0.25, 0.55], pass_zero=True, window="hamming"),
}

plt.figure(figsize=(11, 5))
for name, coeffs in firwin_filters.items():
    plot_magnitude(coeffs, "Outros filtros FIR com firwin - modulo em dB", db=True, label=name)
plt.ylim(-120, 5)
plt.legend()
plt.tight_layout()
plt.show()


## Item 4 - Estimativa da ordem com janela Kaiser

Especificacoes:

- $F_p = 1.8$ kHz
- $F_{stop} = 2.0$ kHz
- $F_s = 12$ kHz
- Atenuacao minima = 35 dB

A largura de transicao e:

$$\Delta f = F_{stop} - F_p = 200\text{ Hz}$$

Normalizando pela frequencia de Nyquist:

$$width = \frac{200}{12000/2} = 0.0333$$


In [ ]:
fp = 1800
fstop = 2000
fs_kaiser = 12000
attenuation_db = 35

transition_width = fstop - fp
width = transition_width / (fs_kaiser / 2)
numtaps_kaiser, beta = signal.kaiserord(attenuation_db, width)

if numtaps_kaiser % 2 == 0:
    numtaps_kaiser += 1

cutoff = (fp + fstop) / 2
h_kaiser_lowpass = signal.firwin(numtaps_kaiser, cutoff, window=("kaiser", beta), fs=fs_kaiser)

print(f"Largura de transicao: {transition_width} Hz")
print(f"Width normalizado: {width:.4f}")
print(f"Numero de coeficientes estimado: {numtaps_kaiser}")
print(f"Ordem classica estimada: {numtaps_kaiser - 1}")
print(f"Beta da Kaiser: {beta:.3f}")
print(f"Frequencia de corte usada no projeto: {cutoff} Hz")

plt.figure(figsize=(11, 5))
w, response = signal.freqz(h_kaiser_lowpass, worN=4096, fs=fs_kaiser)
plt.plot(w, 20 * np.log10(np.maximum(np.abs(response), 1e-8)))
plt.axvline(fp, color="g", linestyle="--", label="Fp")
plt.axvline(fstop, color="r", linestyle="--", label="Fstop")
plt.title("Passa-baixa FIR com janela Kaiser")
plt.xlabel("Frequencia (Hz)")
plt.ylabel("Modulo (dB)")
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## Item 5 - Janela Kaiser com diferentes parametros

O parametro `beta` controla o compromisso da janela Kaiser. Valores maiores de `beta` reduzem os lobulos laterais, aumentando a atenuacao, mas deixam a transicao mais larga para um mesmo numero de coeficientes.

In [ ]:
numtaps = 101
cutoff_norm = 0.3
betas = [0.0, 5.0, 8.6, 14.0]

plt.figure(figsize=(11, 5))
for beta_value in betas:
    h = signal.firwin(numtaps, cutoff_norm, window=("kaiser", beta_value))
    plot_magnitude(h, "Passa-baixa com janela Kaiser - diferentes beta", db=True, label=f"beta={beta_value}")
plt.ylim(-140, 5)
plt.legend()
plt.tight_layout()
plt.show()


## Kaiser em filtros passa-alta e passa-banda

Os proximos exemplos usam Kaiser para filtros passa-alta e passa-banda.

In [ ]:
numtaps = 101
beta_value = 8.6

kaiser_highpass = signal.firwin(numtaps, 0.45, pass_zero=False, window=("kaiser", beta_value))
kaiser_bandpass = signal.firwin(numtaps, [0.25, 0.55], pass_zero=False, window=("kaiser", beta_value))

plt.figure(figsize=(11, 5))
plot_magnitude(kaiser_highpass, "Filtros FIR com janela Kaiser", db=True, label="passa-alta")
plot_magnitude(kaiser_bandpass, "Filtros FIR com janela Kaiser", db=True, label="passa-banda")
plt.ylim(-140, 5)
plt.legend()
plt.tight_layout()
plt.show()


## Alterando a atenuacao e calculando a ordem

Mantendo a mesma largura de transicao, aumentar a atenuacao exigida aumenta o numero de coeficientes do filtro.

In [ ]:
for attenuation in [35, 50, 60, 80]:
    taps, beta_est = signal.kaiserord(attenuation, width)
    if taps % 2 == 0:
        taps += 1
    print(f"Atenuacao = {attenuation:>2} dB | coeficientes = {taps:>4} | ordem = {taps - 1:>4} | beta = {beta_est:.3f}")


## Filtro FIR rejeita-banda (notch) com Kaiser

Um filtro notch e um rejeita-banda estreito. Ele e usado para remover uma faixa pequena de frequencias, por exemplo uma interferencia concentrada em torno de uma frequencia especifica, sem afetar muito o restante do sinal.

Abaixo foi projetado um notch em torno de 2 kHz, removendo aproximadamente a faixa entre 1.9 kHz e 2.1 kHz.

In [ ]:
fs_notch = 12000
notch_band = [1900, 2100]
attenuation_db = 60
transition_hz = 150
width_notch = transition_hz / (fs_notch / 2)
numtaps_notch, beta_notch = signal.kaiserord(attenuation_db, width_notch)
if numtaps_notch % 2 == 0:
    numtaps_notch += 1

h_notch = signal.firwin(
    numtaps_notch,
    notch_band,
    pass_zero="bandstop",
    window=("kaiser", beta_notch),
    fs=fs_notch,
)

print(f"Notch de {notch_band[0]} Hz a {notch_band[1]} Hz")
print(f"Coeficientes: {numtaps_notch}")
print(f"Ordem classica: {numtaps_notch - 1}")
print(f"Beta: {beta_notch:.3f}")

plt.figure(figsize=(11, 5))
w, response = signal.freqz(h_notch, worN=4096, fs=fs_notch)
plt.plot(w, 20 * np.log10(np.maximum(np.abs(response), 1e-8)))
plt.axvspan(notch_band[0], notch_band[1], color="r", alpha=0.15, label="faixa rejeitada")
plt.title("Filtro FIR notch com janela Kaiser")
plt.xlabel("Frequencia (Hz)")
plt.ylabel("Modulo (dB)")
plt.ylim(-120, 5)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()
